# LLM Agents

## Goal of this Lab

The aim of this lab is to build a system that reviews academic papers, and to see whether splitting that job across several LLM agents beats asking a single model once. We start from a plain prompt, add personas, then let those personas talk to each other across multiple rounds. Along the way we look at what all of this costs, and how to bring that cost down. In short, this lab is split into two parts.

- A peer-review pipeline, from single-prompt baseline to multi-agent discussion
- Context compression, and what it does to the quality of the output

## Introduction

You wake up. The deadline for the paper submission is tomorrow. You have gone over the manuscript again and again, but a lingering fear remains. What if you missed something? What if something gets misinterpreted?

You get out of bed and open ChatGPT. You upload the manuscript and ask for its opinion. Is it enough? Will it catch what the reviewers would? What if one of the reviewers is from a different domain? Will they misunderstand something that the LLM doesn't?

...Is there a better way to do this?

---

### What we will be doing

In this workshop you will build a system that helps researchers review their own work. This system will at first be composed of just prompting an LLM, but will eventually evolve into a multi-agent system. We will be modeling this system after the real-life peer-review process, as it is a tried-and-tested process of combining expert opinions in a formal setting.

By the end you will have a working Python script that simulates structured academic peer review through multiple LLM agents - and you will have seen whether that produces better critiques than a single prompt.

---

### Scope

Loading and prompting models is not included in the scope of this lab. We will instead be using a [specialized library](https://github.com/dimits-ts/syndisco), which abstracts away loading, prompting, and organizing LLM agents. If you want to further explore the concepts introduced in this lab, you could consult the [online documentation](https://dimits-ts.github.io/syndisco), although this will not be necessary for completion.

---

### Objectives

By the end of this session you will be able to:

1. **Design** a role-based multi-agent discussion system to solve a specific problem
2. **Implement** a multi-round interaction loop using Python
3. **Compare** single-prompt vs multi-agent approaches
4. **Justify** for what tasks synthetic discussion is appropriate

---

### Prerequisites

- Basic Python knowledge
- Familiarity with LLMs (basic prompting and terminology)

---

### Disclaimer

This lab does not take any position on the use of LLM agents in real-world peer-review processes. The code provided in this notebook should not be blindly used in any formal review workflow. Before applying LLM agents to peer review, please consult the authorship and review policies of the relevant publisher or organization. Additionally, this lab is not intended to replace traditional literature reviews.

We note that tools exploring similar ideas already exist, such as the [Stanford paper review system](https://paperreview.ai/), and that LLM-assisted reviewing is already being explored experimentally in some peer-review settings, including the [AAAI](https://arxiv.org/abs/2604.13940) and [EMNLP](https://2026.emnlp.org/ai-reviewing-experiment/) AI-Assisted Peer Review Pilots. However, the peer-review scenario in this lab is presented solely as an example of how multiple Points of View (POVs) can be applied to real-world problems and does not reflect the opinions of the volunteering team or the organizers.

The code in this lab is licensed under the [AGPL (GNU Affero Public License) version 3-or-later](https://www.gnu.org/licenses/agpl-3.0.txt).

## Setup

In [1]:
!pip install syndisco --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.4/115.4 kB 10.4 MB/s eta 0:00:00


We will be using locally hosted models for this workshop. The use of local models constrains us by allowing only comparatively less capable LLMs, and reducing how much text we can provide as prompts.

While the `syndisco` library also allows the use of OpenAI models, they are expensive, require preparation (acquiring OpenAI tokens, familiarization with the pricing policy etc.), and are not as accessible. However, feel free to use an OpenAI model by replacing the `TransformersModel` with a `OpenAIModel` from the library.

In [2]:
import syndisco
from syndisco import TransformersModel


# You may want to change this depending on your hardware and needs.
# We recommend picking models from huggingface: https://huggingface.co
model = TransformersModel(
    model_path="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    name="base_model",
    max_out_tokens=1500,
)
syndisco.logging_setup(
    print_to_terminal=True,
    write_to_file=False,
    level="warning",
    use_colors=True
)

config.json:   0%|          | 0.00/1.34k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.05GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

## Task: Reviewing a paper

Before we get into multi-agent systems, let's start simple.

Below is a short excerpt from a research paper. Your task is to review it using a language model. You can prompt the model however you like-there are no constraints yet. Treat this as an initial experiment. Run the cells as they are first, then start changing things: make the reviewer harsher, or narrower, or ask it for a specific output format.

**Note**: We are only providing the abstract of a very famous paper, which all LLMs should know by heart from their pretraining. Providing the whole paper may lead to truncation or an Out Of Memory (OOM) error in local models. You can run this lab as-is and be confident that the LLMs react as if they had the entire paper available to them - however, if you want to insert a less-known paper, you should find a way to shrink the input (maybe through summarization).

In [3]:
# --- Paper abstract to review -----------------------------------------------
ABSTRACT = """
Title: Attention Is All You Need

Abstract:
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and
convolutions entirely. Experiments on two machine translation tasks show these
models to be superior in quality while being more parallelizable and requiring
significantly less time to train.
"""

Prompting is fairly simple. We just need to define a system prompt (to clue our model in what the task is), and a user prompt (which contains the actual "meat" of our request).

In [4]:
BASELINE_SYSTEM = "You are an expert academic reviewer. Review the paper excerpt provided."
BASELINE_USER   = f"Please review the following paper abstract:\n\n{ABSTRACT}"

baseline_review = model.prompt(BASELINE_SYSTEM, BASELINE_USER)

print("=" * 60)
print("BASELINE — single-prompt review")
print("=" * 60)
print(baseline_review)

[transformers] Both `max_new_tokens` (=1500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


BASELINE — single-prompt review
The provided abstract for the paper titled "Attention Is All You Need" is quite clear and concise, but it could benefit from some minor adjustments to enhance its clarity and impact. Here is a revised version:

---

**Title:** Attention Is All You Need

**Abstract:**
Recent sequence-to-sequence models often rely on complex architectures involving recurrent or convolutional neural networks (RNNs/CNNs) with an encoder-decoder structure. These models typically incorporate an attention mechanism to improve performance. In this work, we introduce the Transformer, a novel network architecture that leverages only attention mechanisms without the need for RNNs or CNNs. Our experiments demonstrate that the Transformer achieves superior performance on two machine translation tasks, while being more parallelizable and requiring significantly less training time.

---

Key points of improvement:
1. **Clarity:** The abstract now clearly states the problem (complex arc

There is of course no automated way of evaluating this review. What we can do is have a look at the output ourselves, and determine some task-specific quality checks. Since our task is academic reviewing these checks could look something like this:

---

* [ ] **On track?**
  Does the response actually address the question?

* [ ] **Clear + readable?**
  Is it easy to follow, or a bit messy?

* [ ] **Some real thinking?**
  Does it go beyond obvious or generic points?

* [ ] **Balanced?**
  Does it consider more than one angle, or just present a single take?

* [ ] **Any pushback?**
  Does it question assumptions or just accept them?

* [ ] **Feels complete?**
  Do you feel like you got a useful answer, or something partial/shallow?

* [ ] **Feels realistic?**
  Does this read like an actual human-generated review? Could you discern between the two?

* [ ] **Overall feel**
  Does this feel genuinely helpful for your task?

* [ ] **Most importantly**
  Did you actually get any insights on the paper?
  
*(We'll come back to this exact checklist later)*

---

## Adding spice to the reviews

A simple way of generating more diverse and targeted answers is by adding more details to the system prompt. While this can be trivially achieved by directly changing the system prompt above (try it out!), we can use a standard template to make our life easier when we define multiple LLM paricipants.

Our library already takes care of basic templating using the `Actor` class. Let's define a very skeptical reviewer using the following attributes:
- **`persona`** — a dict of attributes that shape its voice (role, expertise, style)
- **`context`** — what the discussion is about (ideally shared across all agents)
- **`instructions`** — same as before, but can be customized for each agent

In [5]:
from syndisco import Actor


REVIEW_CONTEXT = (
    f"You are participating in a structured academic peer-review discussion. "
    f"The paper under review is:\n\n{ABSTRACT}"
)

REVIEW_INSTRUCTIONS = "Write a full, professional review of the article for a prestigious journal."

skeptical_reviewer = Actor(
    model=model,
    persona={
        "role": "Skeptical Reviewer",
        "expertise": "critical analysis, assumption-testing, identifying gaps",
        "style": "challenging, rigorous, devil's advocate",
    },
    context=REVIEW_CONTEXT,
    instructions=REVIEW_INSTRUCTIONS,
    name="SkepticalReviewer",
)

skeptical_review = skeptical_reviewer.speak()
print("=" * 60)
print("SKEPTICAL — single-prompt review")
print("=" * 60)
print(skeptical_review)

[transformers] Both `max_new_tokens` (=1500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SKEPTICAL — single-prompt review
Given the abstract provided, it appears that the Transformer model stands out as a significant advancement in sequence transduction models, particularly in the realm of machine translation. However, as a skeptical reviewer, I will critically analyze the paper, focusing on its assumptions, experimental robustness, and broader implications.

### Introduction and Background

The paper introduces the Transformer model, which is a novel approach that leverages attention mechanisms without the need for recurrent or convolutional layers. This shift from traditional neural network architectures is indeed intriguing and could potentially lead to more efficient and scalable models. 

### Methodology

The Transformer model is described as a simple yet powerful architecture. It is important to verify whether this simplicity comes at the cost of performance. The authors should provide a thorough comparison with existing models, including their baseline performance m

What have we gained by creating a different persona? Generally speaking, we have a whole other Point of View, which we would have otherwised missed. This is the core motivation of using different agents.



## Task: Multi-agent prompting

Let's spice things up even further. Why not simulate the entire first round of the review?

A normal peer-review takes anywhere from two to four reviewers. We already have a skeptical reviewer, so think about what other *points of view* would be beneficial for the review.

We should also add a **meta-reviewer** as a participant here — they should join at the end of the review cycle, when all other reviews have been posted.

In [6]:
# --- Statistical Reviewer ---------------------------------------------------
statistical_reviewer = Actor(
    model=model,
    persona={
        "role": "Statistical Reviewer",
        "expertise": "statistics, experimental design, causal inference, reproducibility",
        "style": "rigorous, detail-oriented, critical but constructive",
    },
    context=REVIEW_CONTEXT,
    instructions=REVIEW_INSTRUCTIONS,
    name="StatisticalReviewer",
)

# --- NLP Expert Reviewer ----------------------------------------------------
nlp_reviewer = Actor(
    model=model,
    persona={
        "role": "NLP Expert Reviewer",
        "expertise": "natural language processing, deep learning, LLMs, benchmarking",
        "style": "scholarly, literature-aware, technically grounded",
    },
    context=REVIEW_CONTEXT,
    instructions=REVIEW_INSTRUCTIONS,
    name="NLPReviewer",
)

# --- Author -----------------------------------------------------------------
author = Actor(
    model=model,
    persona={
        "role": "Author",
        "expertise": "the submitted paper and its contributions",
        "style": "defensive but professional, transparent, evidence-based",
    },
    context=REVIEW_CONTEXT,
    instructions=(
        "You are the author of the submitted paper. Respond to reviewer comments by:\n"
        "  • Addressing each critique directly and respectfully\n"
        "  • Providing clarifications, additional justification, or acknowledging limitations\n"
        "  • Proposing concrete revisions where appropriate\n"
        "  • Avoiding defensiveness; aim to improve the paper\n"
    ),
    name="Authors",
)

# --- Meta-Reviewer ----------------------------------------------------------
meta_reviewer = Actor(
    model=model,
    persona={
        "role": "Meta-Reviewer",
        "expertise": "synthesis, editorial judgment, final recommendations",
        "style": "balanced, decisive, structured",
    },
    context=REVIEW_CONTEXT,
    instructions=(
        "You have read all previous reviewer comments. Synthesise the discussion "
        "into a final structured judgment. Your output must contain:\n"
        "  • Summary of main strengths\n"
        "  • Summary of main weaknesses\n"
        "  • Overall recommendation (Accept / Major Revision / Reject) with a one-sentence rationale."
    ),
    name="MetaReviewer",
)

print("Agents created:")
for agent in [
    statistical_reviewer,
    nlp_reviewer,
    skeptical_reviewer,
    author,
    meta_reviewer,
]:
    print(f"\t* {agent.get_actor_name()}")

Agents created:
	* StatisticalReviewer
	* NLPReviewer
	* SkepticalReviewer
	* Authors
	* MetaReviewer


In round 1 each of the three reviewers should read the abstract with no prior discussion context.

In [7]:
print("Generating round 1 independent reviews...\n")

round1_reviewers = [
    statistical_reviewer,
    nlp_reviewer,
    skeptical_reviewer,
]

# Each reviewer speaks independently — no history yet
print("Round 1 starting...\n")

opinions = []
agent_names = []

for reviewer in round1_reviewers:
    print(f"{'─' * 60}")
    print(f"[{reviewer.get_actor_name()}]")

    opinion = reviewer.speak()  # runs + returns
    print(opinion)

    opinions.append(opinion)
    agent_names.append(reviewer.get_actor_name())

print("─" * 60)
print("Round 1 complete.")

[transformers] Both `max_new_tokens` (=1500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generating round 1 independent reviews...

Round 1 starting...

────────────────────────────────────────────────────────────
[StatisticalReviewer]


[transformers] Both `max_new_tokens` (=1500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Given the abstract provided, it is clear that the paper introduces a novel neural network architecture called the Transformer, which significantly departs from traditional sequence-to-sequence models by eliminating recurrent and convolutional layers. This approach is grounded in the use of attention mechanisms, which have been shown to be effective in various natural language processing (NLP) tasks.

### Statistical Review

#### 1. **Model Architecture**
The Transformer model is described as a simple network architecture that relies solely on attention mechanisms. This simplicity is a key point of interest, as it contrasts with the complexity of traditional models like LSTM or GRU-based RNNs and CNNs. 

- **Attention Mechanism**: The use of attention mechanisms is a significant innovation. It allows the model to focus on relevant parts of the input sequence, which can lead to improved performance and efficiency. However, the statistical significance of this improvement needs to be rigo

[transformers] Both `max_new_tokens` (=1500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Given the abstract provided, it is clear that the paper titled "Attention Is All You Need" introduces a novel neural network architecture called the Transformer, which significantly departs from traditional sequence-to-sequence models by eliminating recurrent and convolutional layers. Instead, the Transformer relies solely on attention mechanisms to process and generate sequences. This approach promises improved performance, enhanced parallelizability, and reduced training times.

### Review of "Attention Is All You Need"

#### Introduction

The paper "Attention Is All You Need" by Vaswani et al. (2017) presents a groundbreaking contribution to the field of natural language processing (NLP) and machine translation. The authors argue that the current state-of-the-art sequence transduction models, such as those using recurrent neural networks (RNNs) or convolutional neural networks (CNNs), are overly complex and resource-intensive. In contrast, the Transformer model, which is based solel

Our pipeline so far should encourage the presence of multiple POVs. Let's revisit the checklist from before, and determine whether utilizing multiple agents made a difference.

---

* [ ] **On track?**
  Does the response actually address the question?

* [ ] **Clear + readable?**
  Is it easy to follow, or a bit messy?

* [ ] **Some real thinking?**
  Does it go beyond obvious or generic points?

* [ ] **Balanced?**
  Does it consider more than one angle, or just present a single take?

* [ ] **Any pushback?**
  Does it question assumptions or just accept them?

* [ ] **Feels complete?**
  Do you feel like you got a useful answer, or something partial/shallow?

* [ ] **Feels realistic?**
  Does this read like an actual human-generated review? Could you discern between the two?

* [ ] **Overall feel**
  Does this feel genuinely helpful for your task?

* [ ] **Most importantly**
  Did you actually get any insights on the paper?
  
We should also add another very important question now:

* [ ] **Was it worth the trouble**?
  Using multiple agents meant spending some time designing them and waiting for their responses. Were the results you got worth the cost in time (and perhaps resources)? Why?

---

## Task: Round 2 and Round 3 — Discussion and meta-review

You have perhaps identified a limitation in our approach so far; the agents respond in isolation, and can not take into account what the other agents have highlighted. This limitation is important for two reasons:

1. The review process after this point will not be **realistic**. In reality, after the first round of reviewing, the authors engage with the reviewers in order to respond to their questions.
    - One of the reasons for building this reviewing pipeline is to simulate what could (not *would*!) happen in the real world. In this case, our setup is a special case of Agent-Based Modeling (ABM). ABM concerns itself with building simulated agents with simple rules, then allowing them to interact with each other to simulate complex human behavior. In our case, since our agents don't need explicit rules, but rather instructions, we are talking abouut Generative Agent-Based Modeling (GABM).
    - A motivation for using synthetic discussions then is to simulate what could have happened in a peer-review process. If you want to explore this part of the discussion, try it out by *adding your own paper to the top of the notebook* and re-running it.

2. The review process will not be **informative**. We can already see that agents may focus on different aspects of the work. Why shouldn't a critical reviewer take into consideration points the strengths mentioned by other reviewers?
    - Going back-and-forth would allow agents to iteratively bring up, discard, and fight over specific points raised in the paper. This process is dynamic, and requires no input from us.
    - If you want to explore what *information* you can extract using synthetic discussions, an abstract may not be enough. Why don't we try loading a whole paper? This will raise some issues, since we are using a local model. How would you circumvent them?

---

We will create a discussion between the reviewers defined above. You could also add an author; an agent that will attempt to defend the paper. How would you defend your paper?

In [8]:
from syndisco import Discussion, QueueTurnManager

all_agents = [
    statistical_reviewer,
    nlp_reviewer,
    skeptical_reviewer,
    author,
    meta_reviewer,
]

# QueueTurnManager cycles through actors in the order they are given.
turn_manager = QueueTurnManager(actors=all_agents)

discussion = Discussion(
    next_turn_manager=turn_manager,
    users=all_agents,
    # In synthetic discussions, initial starting points (hardcoded messages)
    # are usually referred to as "seed opinions"
    # Round-1 reviews are injected as seeds since the reviewers should
    # now know the results of the first round of reviewing
    seed_opinions=opinions,
    seed_opinion_usernames=agent_names,
    # Each agent sees the last 6 messages as context (enough for all seeds + some turns)
    # You may need to turn this down if you are running this notebook in a machine
    # with relatively low VRAM, or if you decide to load a large document.
    history_context_len=6,
    # 3 updated critiques (round 2) + 1 meta-review (round 3)
    conv_len=4,
)

print("Running peer-review rounds 2 and 3...")
discussion.begin(verbose=True)

Running peer-review rounds 2 and 3...


  0%|          | 0/4 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=1500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Comment by user StatisticalReviewer: "
{"context": "Your comment:", "content": "Given the abstract and the initial comments, the Transformer model presented in the paper "Attention Is All You Need" is a significant advancement in the field of sequence transduction models, particularly in machine translation. The model's simplicity and improved performance on translation tasks are compelling arguments for its potential widespread adoption. However, the initial reviews highlight several important aspects that need further attention to ensure the robustness and credibility of the findings. Here are some key points and recommendations for the authors to consider:  ### Key Points: 1. **Statistical Rigor**: The initial reviews emphasize the need for detailed statistical analyses to quantify the improvements in performance. This includes effect size calculations, paired t-tests, and ANOVA comparisons to compare the Transformer model with baseline models.  2. **Causal Inference**: There is a c

[transformers] Both `max_new_tokens` (=1500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Comment by user NLPReviewer: "
Given the abstract and the initial comments, the Transformer model presented in the paper "Attention Is All You Need" is indeed a significant advancement in the field of sequence transduction models, particularly in machine translation. The model's simplicity and improved performance on translation tasks are compelling arguments for its potential widespread adoption. However, the initial reviews highlight several important aspects that need further attention to ensure the robustness and credibility of the findings.  ### Key Points:  1. **Statistical Rigor**: The initial reviews emphasize the need for detailed statistical analyses to quantify the improvements in performance. This includes effect size calculations, paired t-tests, and ANOVA comparisons to compare the Transformer model with baseline models.     2. **Causal Inference**: There is a call for a more rigorous approach using causal inference methods to establish the causal relationship between the

[transformers] Both `max_new_tokens` (=1500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Comment by user SkepticalReviewer: "
Your comment provides a thoughtful and balanced critique of the Transformer model presented in the paper "Attention Is All You Need." Here are some refined suggestions to enhance the robustness and credibility of the findings:  ### Key Points: 1. **Statistical Rigor**:     - Perform detailed statistical analyses to quantify the improvements in performance. This includes effect size calculations, paired t-tests, and ANOVA comparisons to compare the Transformer model with baseline models.     2. **Causal Inference**:    - Use advanced statistical techniques to establish causality between the attention mechanism and performance improvements. This will strengthen the argument for the Transformer model’s superiority.  3. **Generalizability**:    - Conduct cross-task experiments to evaluate the Transformer model’s generalizability. This is crucial to determine its broad applicability across different NLP tasks.  4. **Experimental Robustness**:    - Provid

## Was it worth it?

Let's print the baseline single-prompt review alongside the meta-reviewer's final judgment.

In [9]:
# The meta-reviewer is always the last entry in the log
meta_review_entry = discussion.get_logs()[-1]

divider = "=" * 60

print(divider)
print("BASELINE — single-prompt review")
print(divider)
print(baseline_review)

print()
print(divider)
print(f"MULTI-AGENT — {meta_review_entry['name']} final judgment")
print(divider)
print(meta_review_entry['text'])

BASELINE — single-prompt review
The provided abstract for the paper titled "Attention Is All You Need" is quite clear and concise, but it could benefit from some minor adjustments to enhance its clarity and impact. Here is a revised version:

---

**Title:** Attention Is All You Need

**Abstract:**
Recent sequence-to-sequence models often rely on complex architectures involving recurrent or convolutional neural networks (RNNs/CNNs) with an encoder-decoder structure. These models typically incorporate an attention mechanism to improve performance. In this work, we introduce the Transformer, a novel network architecture that leverages only attention mechanisms without the need for RNNs or CNNs. Our experiments demonstrate that the Transformer achieves superior performance on two machine translation tasks, while being more parallelizable and requiring significantly less training time.

---

Key points of improvement:
1. **Clarity:** The abstract now clearly states the problem (complex arc

Is the meta-review sufficient? Should we also include the discussion itself in our analysis? If so, should we keep the entire discussion? Should we summarize it?

These are all practical questions which you will need to answer depending on the task, what you are looking for, whether you want to maximize information extraction or simulate many peer-review processes, etc.

Let's now refer to the checklist one last time.

---

* [ ] **On track?**
  Does the response actually address the question?

* [ ] **Clear + readable?**
  Is it easy to follow, or a bit messy?

* [ ] **Some real thinking?**
  Does it go beyond obvious or generic points?

* [ ] **Balanced?**
  Does it consider more than one angle, or just present a single take?

* [ ] **Any pushback?**
  Does it question assumptions or just accept them?

* [ ] **Feels complete?**
  Do you feel like you got a useful answer, or something partial/shallow?

* [ ] **Feels realistic?**
  Does this read like an actual human-generated review? Could you discern between the two?

* [ ] **Overall feel**
  Does this feel genuinely helpful for your task?

* [ ] **Most importantly**
  Did you actually get any insights on the paper?
  
* [ ] **Was it worth the trouble**?
  Were the results you got worth the cost in time (and perhaps resources)? Why?

---

## To summarize

So far you:

1. Attempted to solve a problem through a traditional **single-prompt** approach
1. Used instructions and personas to build an alternative **role-specialised multi-agent** system
1. Modeled the specific problem in terms of **interactions** between multiple agents
1. Used **synthetic discussions** to manage context, combine information, and dynamically shift the focus of the agents around different parts of the paper

Having run the four different setups, you now should have an intuition on the effort, time and resources needed for each LLM-led approach. Given a new research problem, you should be able to determine which of these approaches is appropriate, and how you can transform it into an appopriate, agent-based approach.

## Context Management

You now have a working multi-agent system. The natural next step is to throw bigger problems at it. Longer papers, more reviewers, more discussion rounds.

This is where things start breaking.

---

In practice, when people deploy autonomous agents, they sometimes let them run overnight and wake up to API bills in the thousands of euros. More rounds mean more tokens, and tokens cost money.

But cost is not the only concern. Time scales with input length too. A discussion that takes 2 minutes on an abstract could take 20 on a full paper, assuming it runs at all. If you are using open-source models locally (like we are), you are practically limited by how much context your hardware can fit. Go over that limit and you get truncation, or an out-of-memory crash.

There is also a subtler problem. As the context grows, models start losing track of what matters. Important points from early in the discussion get buried under newer, less relevant text. The output gets vaguer, more generic, less useful. Exactly the opposite of what we set out to do.

---

So: can we keep the benefits of a multi-agent discussion while spending fewer tokens to get there?

Natural language is redundant. We pad our writing with grammar, filler, and structure that humans need to parse text, but that LLMs can reconstruct on their own. If we strip that padding before feeding text into a model, we can fit more meaningful content into the same context window.

This is the idea behind **text compression** for LLMs, and we will explore some ways of doing it.

### Task: Caveman Compression

Who would have thought that one way forward in NLP involves writing like a caveman?

The idea is embarrassingly simple. LLMs are trained on trillions of tokens of well-formed text. They are very good at predicting grammar, filler words, and connectives. So if we remove all of that and keep only the core factual content, the model can still understand the input perfectly well. We save tokens, and the model does the work of reconstructing the rest.

[Caveman Compression](https://github.com/wilpel/caveman-compression) does exactly this. There are ways to do it using masked language models or other LMs to decide what to cut, but for our purposes we will stick with simple text processing: parse the input with spaCy, identify parts of speech, and strip away what the model doesn't need. Determiners, conjunctions, auxiliary verbs, grammatical glue. What's left reads like a telegram, but carries the same meaning.

Let's see what it does to our abstract.

In [10]:
# setup
!git clone https://github.com/wilpel/caveman-compression.git --quiet
!pip install spacy --quiet
!python -m spacy download en_core_web_sm --quiet

import sys
sys.path.append("caveman-compression")
from caveman_compress_nlp import compress_text as caveman_compress

# compress the abstract
compressed_abstract = caveman_compress(ABSTRACT)

print("ORIGINAL:")
print(ABSTRACT)
print()
print("CAVEMAN:")
print(compressed_abstract)
print()
print(f"Original length:   {len(ABSTRACT.split())} words")
print(f"Compressed length: {len(compressed_abstract.split())} words")
print(f"Reduction:         {1 - len(compressed_abstract.split()) / len(ABSTRACT.split()):.0%}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 89.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
ORIGINAL:

Title: Attention Is All You Need

Abstract:
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and
convolutions entirely. Experiments on two machine translation tasks show these
models to be superior in quality while being more parallelizable and requiri

### Measuring the impact

We are about to re-run the full multi-agent review pipeline, but this time with caveman
compression applied to the input and the discussion history. The final meta-review will be
left uncompressed, since that is the output a human will actually read.

Before we do that, we need to decide what to measure. We will track three things:

1. **Tokens used** — how much context did we save?
3. **[BERTScore](https://github.com/Tiiiger/bert_score)** — how much did the output *change* compared to the uncompressed version?
4. **The checklist** — is the output actually good? This one is on you.

A note on BERTScore. We are not using it as a quality metric here. We have no ground truth.
What BERTScore tells us is how *similar* the compressed meta-review is to the uncompressed
one. High similarity means compression did not change much. Low similarity means it did.
Whether that change is good or bad is for you to judge.

In [11]:
# some utility functions
import time

!pip install bert_score --quiet

from bert_score import score as bert_score
from transformers import AutoTokenizer
import logging

# Suppress the Hugging Face Transformers logging to only show ERROR messages
logging.getLogger("transformers").setLevel(logging.ERROR)

# We use the same tokenizer as our model for accurate token counts
tokenizer = AutoTokenizer.from_pretrained("unsloth/Qwen2.5-3B-Instruct-bnb-4bit")


def count_tokens(text):
    return len(tokenizer.encode(text))


def compute_bertscore(candidate, reference):
    """Compare two texts. Returns precision, recall, F1."""
    P, R, F1 = bert_score(
        [candidate], [reference], lang="en", verbose=False, device="cpu"
    )
    return {"precision": P.item(), "recall": R.item(), "f1": F1.item()}

# Agent creation function. This is a copy of the original one since we don't want to be pasting this entire thing in multiple cells.
# If you want to change agent behaviors, you can just do it here in the persona part.
def create_agents(context):
    reviewers = [
        Actor(
            model=model,
            persona={
                "role": "Statistical Reviewer",
                "expertise": "statistics, experimental design, causal inference, reproducibility",
                "style": "rigorous, detail-oriented, critical but constructive",
            },
            context=context,
            instructions=REVIEW_INSTRUCTIONS,
            name="StatisticalReviewer",
        ),
        Actor(
            model=model,
            persona={
                "role": "NLP Expert Reviewer",
                "expertise": "natural language processing, deep learning, LLMs, benchmarking",
                "style": "scholarly, literature-aware, technically grounded",
            },
            context=context,
            instructions=REVIEW_INSTRUCTIONS,
            name="NLPReviewer",
        ),
        Actor(
            model=model,
            persona={
                "role": "Skeptical Reviewer",
                "expertise": "critical analysis, assumption-testing, identifying gaps",
                "style": "challenging, rigorous, devil's advocate",
            },
            context=context,
            instructions=REVIEW_INSTRUCTIONS,
            name="SkepticalReviewer",
        ),
    ]

    author = Actor(
        model=model,
        persona={
            "role": "Author",
            "expertise": "the submitted paper and its contributions",
            "style": "defensive but professional, transparent, evidence-based",
        },
        context=context,
        instructions=(
            "You are the author of the submitted paper. Respond to reviewer comments by:\n"
            "  • Addressing each critique directly and respectfully\n"
            "  • Providing clarifications, additional justification, or acknowledging limitations\n"
            "  • Proposing concrete revisions where appropriate\n"
            "  • Avoiding defensiveness; aim to improve the paper\n"
        ),
        name="Authors",
    )

    meta = Actor(
        model=model,
        persona={
            "role": "Meta-Reviewer",
            "expertise": "synthesis, editorial judgment, final recommendations",
            "style": "balanced, decisive, structured",
        },
        context=context,
        instructions=(
            "You have read all previous reviewer comments. Synthesise the discussion "
            "into a final structured judgment. Your output must contain:\n"
            "  • Summary of main strengths\n"
            "  • Summary of main weaknesses\n"
            "  • Overall recommendation (Accept / Major Revision / Reject) with a one-sentence rationale."
        ),
        name="MetaReviewer",
    )

    return reviewers, author, meta


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.6 MB/s eta 0:00:00


In [12]:
compressed_context = (
    f"You are participating in a structured academic peer-review discussion. "
    f"The paper under review is:\n\n{caveman_compress(ABSTRACT)}"
)

compressed_reviewers, compressed_author, compressed_meta = create_agents(compressed_context)

# Count the compressed context sent to each reviewer
total_tokens = count_tokens(compressed_context) * 3

caveman_opinions = []
caveman_agent_names = []

for reviewer in compressed_reviewers:
    opinion = reviewer.speak()
    compressed_opinion = caveman_compress(opinion)
    caveman_opinions.append(compressed_opinion)
    caveman_agent_names.append(reviewer.get_actor_name())
    # Output tokens + compressed opinion that becomes a seed
    total_tokens += count_tokens(opinion) + count_tokens(compressed_opinion)
    print(f"[{reviewer.get_actor_name()}] done")

# Rounds 2 and 3: discussion using compressed seed opinions
all_compressed_agents = compressed_reviewers + [compressed_author, compressed_meta]

caveman_discussion = Discussion(
    next_turn_manager=QueueTurnManager(actors=all_compressed_agents),
    users=all_compressed_agents,
    seed_opinions=caveman_opinions,
    seed_opinion_usernames=caveman_agent_names,
    history_context_len=6,
    conv_len=4,
)

caveman_discussion.begin(verbose=True)

for entry in caveman_discussion.get_logs():
    total_tokens += count_tokens(entry["text"])

caveman_meta_review = caveman_discussion.get_logs()[-1]["text"]
caveman_total_tokens = total_tokens

[StatisticalReviewer] done
[NLPReviewer] done
[SkepticalReviewer] done


  0%|          | 0/4 [00:00<?, ?it/s]

Comment by user StatisticalReviewer: "
Based on the provided abstract and the comments from the reviewers, the paper presents an innovative approach to sequence transduction using a Transformer-based network architecture that dispenses with traditional recurrent and convolutional layers. This approach is particularly noteworthy given the growing interest in Transformer architectures for handling sequential data tasks such as machine translation.  ### Statistical Review  #### 1. **Model Architecture** The proposed model leverages a Transformer-based architecture, which is known for its effectiveness in handling sequential data. It is crucial to evaluate whether the improvements observed are statistically significant and not due to random chance. Specifically, the authors should conduct rigorous statistical tests to validate the performance gains over existing methods. Pairwise t-tests or ANOVA could be used to determine if the differences in performance are statistically significant.  #

In [13]:
baseline_meta = meta_review_entry["text"]

# Count the context sent to each reviewer
baseline_total_tokens = count_tokens(REVIEW_CONTEXT) * 3
for opinion in opinions:
    # Output tokens + same opinion reused as seed input
    baseline_total_tokens += count_tokens(opinion) * 2
for entry in discussion.get_logs():
    baseline_total_tokens += count_tokens(entry["text"])

bertscore_result = compute_bertscore(caveman_meta_review, baseline_meta)

divider = "=" * 60
print(divider)
print("CAVEMAN — final review")
print(divider)
print(caveman_meta_review)
print()
print(f"Tokens (baseline): {baseline_total_tokens}")
print(f"Tokens (caveman):  {caveman_total_tokens}")
print(f"BERTScore F1:      {bertscore_result['f1']:.4f}")

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

CAVEMAN — final review
Based on the feedback from the reviewers, I appreciate the constructive criticism and will take the following actions to enhance the paper:

### Addressing Critiques Directly:
1. **Model Architecture**
   - I will provide detailed descriptions of the attention mechanisms and the modifications made to the standard Transformer architecture.
   - I will also include additional datasets and tasks to validate the model's robustness across different domains.

2. **Experiments and Results**
   - I will include a comprehensive comparison with state-of-the-art models, including metrics like BLEU, METEOR, and ROUGE scores.
   - I will include a control group where the Transformer model is trained without attention mechanisms to isolate the effect of the attention mechanisms.

3. **Reproducibility**
   - I will clearly document the steps required to reproduce the experiments, including the exact versions of the Transformer library, hyperparameters, and relevant software dep

Let's revisit the checklist one more time, this time for the caveman-compressed pipeline.

---

* [ ] **On track?**
  Does the response actually address the question?

* [ ] **Clear + readable?**
  Is it easy to follow, or a bit messy?

* [ ] **Some real thinking?**
  Does it go beyond obvious or generic points?

* [ ] **Balanced?**
  Does it consider more than one angle, or just present a single take?

* [ ] **Any pushback?**
  Does it question assumptions or just accept them?

* [ ] **Feels complete?**
  Do you feel like you got a useful answer, or something partial/shallow?

* [ ] **Feels realistic?**
  Does this read like an actual human-generated review? Could you discern between the two?

* [ ] **Overall feel**
  Does this feel genuinely helpful for your task?

* [ ] **Most importantly**
  Did you actually get any insights on the paper?

---

You probably noticed that the token savings were modest. That is because we only compressed the
*inputs* — the abstract and the seed opinions. The agents still generated full-length responses,
which is where most of the tokens come from. Compression only helps again when those responses
become input for the next agent in the discussion.

How could you save even more? Think about what else you could compress. We compressed agent
outputs after they were generated, but the generation cost was already spent. Could you use
prompting to affect how much the agents produce in the first place?

### Task: Selective Context with LLMLingua

Caveman compression relies on fixed rules. It does not care whether a word is important for
the specific text it appears in. It just checks the part of speech and decides.

Can we do better?

[LLMLingua](https://github.com/microsoft/LLMLingua) takes a different approach. Instead of
rules, it uses a small language model to score each token by how important it is in context.
Tokens that carry less information get dropped. The ones that are surprising or essential
stay.

The intuition is the same as caveman: natural language is redundant, and LLMs can fill in
the gaps. But instead of us deciding what is redundant, we let a model figure it out. This
tends to produce more aggressive compression, and the output is often barely readable to
humans. That is fine. The target audience is not us.

Let's see how it compares.

In [14]:
!pip install llmlingua --quiet

from llmlingua import PromptCompressor

# cpu cause we're tight on VRAM plus it's a small model
llm_lingua = PromptCompressor(
    model_name="microsoft/llmlingua-2-xlm-roberta-large-meetingbank",
    use_llmlingua2=True,
    device_map="cpu",
)

# Compress the abstract
# rate stands for the "keep" rate which is 1 - compression. So if you wanted to compress 30% of the text, you'd write rate = 0.7
# you can try to play around with these rates and see how it compares to other rates and also caveman mode.
lingua_result = llm_lingua.compress_prompt(ABSTRACT, rate=0.5)
lingua_abstract = lingua_result["compressed_prompt"]

print("ORIGINAL:")
print(ABSTRACT)
print()
print("LLMLINGUA:")
print(lingua_abstract)
print()
print(f"Original length:   {len(ABSTRACT.split())} words")
print(f"Compressed length: {len(lingua_abstract.split())} words")
print(f"Reduction:         {1 - len(lingua_abstract.split()) / len(ABSTRACT.split()):.0%}")

config.json:   0%|          | 0.00/752 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.24GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

ORIGINAL:

Title: Attention Is All You Need

Abstract:
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and
convolutions entirely. Experiments on two machine translation tasks show these
models to be superior in quality while being more parallelizable and requiring
significantly less time to train.


LLMLINGUA:
Attention Need dominant sequence transduction models complex neural networks encoder decoder best models connect encoder decoder through attention mechanism propose new network architecture Transformer attention mechanisms dispensing recurrence convolutions Experiments two machine translation tasks show models superior quality more parallelizable less time train

O

In [15]:
# Wrap LLMLingua in a simple function like caveman_compress
# rate is the "keep" rate: rate=0.5 means keep 50% of tokens, so 50% compression
def lingua_compress(text):
    result = llm_lingua.compress_prompt(text, rate=0.5)
    return result["compressed_prompt"]

# Compress the abstract and create agents
lingua_context = (
    f"You are participating in a structured academic peer-review discussion. "
    f"The paper under review is:\n\n{lingua_compress(ABSTRACT)}"
)

lingua_reviewers, lingua_author, lingua_meta = create_agents(lingua_context)

# Count the compressed context sent to each reviewer
total_tokens = count_tokens(lingua_context) * 3

lingua_opinions = []
lingua_agent_names = []

for reviewer in lingua_reviewers:
    opinion = reviewer.speak()
    compressed_opinion = lingua_compress(opinion)
    lingua_opinions.append(compressed_opinion)
    lingua_agent_names.append(reviewer.get_actor_name())
    # Output tokens + compressed opinion that becomes a seed
    total_tokens += count_tokens(opinion) + count_tokens(compressed_opinion)
    print(f"[{reviewer.get_actor_name()}] done")

# Rounds 2 and 3
all_lingua_agents = lingua_reviewers + [lingua_author, lingua_meta]

lingua_discussion = Discussion(
    next_turn_manager=QueueTurnManager(actors=all_lingua_agents),
    users=all_lingua_agents,
    seed_opinions=lingua_opinions,
    seed_opinion_usernames=lingua_agent_names,
    history_context_len=6,
    conv_len=4,
)

lingua_discussion.begin(verbose=True)

for entry in lingua_discussion.get_logs():
    total_tokens += count_tokens(entry["text"])

lingua_meta_review = lingua_discussion.get_logs()[-1]["text"]
lingua_total_tokens = total_tokens

[StatisticalReviewer] done
[NLPReviewer] done
[SkepticalReviewer] done


  0%|          | 0/4 [00:00<?, ?it/s]

Comment by user StatisticalReviewer: "
Based on the comments provided by the reviewers, it is clear that the paper presents a significant advancement in the field of sequence transduction models, particularly through its innovative integration of attention mechanisms within the Transformer architecture. Here are some key points and suggestions for further refinement:  ### Introduction The introduction effectively sets the stage by highlighting the limitations of traditional recurrent and convolutional neural networks (RNNs and CNNs) in handling complex sequential data, especially in tasks like machine translation. It emphasizes the necessity for a better approach and the role of attention mechanisms in capturing dependencies across long sequences.  ### Methodology The methodology section should be expanded to include a more detailed explanation of the proposed network architecture. Specifically, it would be beneficial to: - **Provide a step-by-step description** of how the attention me

In [16]:
lingua_bertscore = compute_bertscore(lingua_meta_review, baseline_meta)

divider = "=" * 60
print(divider)
print("LLMLINGUA — final review")
print(divider)
print(lingua_meta_review)
print()
print(f"Tokens (baseline):  {baseline_total_tokens}")
print(f"Tokens (llmlingua): {lingua_total_tokens}")
print(f"BERTScore F1:       {lingua_bertscore['f1']:.4f}")

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

LLMLINGUA — final review
{"content": "Thank you for your thoughtful feedback. I appreciate the detailed suggestions you have provided, which will indeed help me refine the paper further. Here are my responses to the critiques and suggestions:

### Attention Mechanism Implementation
I will conduct a detailed analysis of the impact of the attention mechanism on different layers of the model. This will involve examining how the attention mechanism affects the quality of the output at various stages of the decoding process. Additionally, I will compare the proposed attention mechanism with other state-of-the-art attention mechanisms to highlight its unique strengths and weaknesses.

### Scalability and Generalization
To ensure that the model's performance generalizes well across different linguistic domains and tasks, I will perform additional experiments on diverse datasets and languages. This will include text summarization and dialogue systems to validate the model's versatility beyond 

Let's revisit the checklist, this time for the LLMLingua-compressed pipeline.

---

* [ ] **On track?**
  Does the response actually address the question?

* [ ] **Clear + readable?**
  Is it easy to follow, or a bit messy?

* [ ] **Some real thinking?**
  Does it go beyond obvious or generic points?

* [ ] **Balanced?**
  Does it consider more than one angle, or just present a single take?

* [ ] **Any pushback?**
  Does it question assumptions or just accept them?

* [ ] **Feels complete?**
  Do you feel like you got a useful answer, or something partial/shallow?

* [ ] **Feels realistic?**
  Does this read like an actual human-generated review? Could you discern between the two?

* [ ] **Overall feel**
  Does this feel genuinely helpful for your task?

* [ ] **Most importantly**
  Did you actually get any insights on the paper?

---

### Task: Extractive Summarization

Caveman compression strips words. LLMLingua strips tokens. Both keep every sentence, just
shorter. What if we go a step further and drop entire sentences instead?

Extractive summarization does exactly that. It scores each sentence by how important it is
to the overall text, keeps the top ones, and throws away the rest. No paraphrasing, no
rewriting. The surviving sentences are untouched originals.

We will use [sumy](https://github.com/miso-belica/sumy), a lightweight library that
implements several extractive methods. We will go with TextRank, which scores sentences
based on how similar they are to the rest of the text. Sentences that overlap a lot with
many others are considered more central and survive.

In [17]:
!pip install sumy --quiet
import nltk
nltk.download('punkt_tab', quiet=True)

from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.text_rank import TextRankSummarizer
from sumy.nlp.stemmers import Stemmer

summarizer = TextRankSummarizer(Stemmer("english"))

def extractive_compress(text, ratio=0.5):
    parser = PlaintextParser.from_string(text, Tokenizer("english"))
    total = len(parser.document.sentences)
    keep = max(1, int(total * ratio))
    summary = summarizer(parser.document, keep)
    return " ".join(str(s) for s in summary)

# Demo on the abstract
extractive_abstract = extractive_compress(ABSTRACT)

print("ORIGINAL:")
print(ABSTRACT)
print()
print("EXTRACTIVE:")
print(extractive_abstract)
print()
print(f"Original length:   {len(ABSTRACT.split())} words")
print(f"Extractive length: {len(extractive_abstract.split())} words")
print(f"Reduction:         {1 - len(extractive_abstract.split()) / len(ABSTRACT.split()):.0%}")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 37.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
ORIGINAL:

Title: Attention Is All You Need

Abstract:
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and
convolutions entirely. Experiments on two machine translation ta

In [18]:
# Compress the abstract and create agents
extractive_context = (
    f"You are participating in a structured academic peer-review discussion. "
    f"The paper under review is:\n\n{extractive_compress(ABSTRACT)}"
)

extractive_reviewers, extractive_author, extractive_meta = create_agents(extractive_context)

# Count the compressed context sent to each reviewer
total_tokens = count_tokens(extractive_context) * 3

extractive_opinions = []
extractive_agent_names = []

for reviewer in extractive_reviewers:
    opinion = reviewer.speak()
    compressed_opinion = extractive_compress(opinion)
    extractive_opinions.append(compressed_opinion)
    extractive_agent_names.append(reviewer.get_actor_name())
    # Output tokens + compressed opinion that becomes a seed
    total_tokens += count_tokens(opinion) + count_tokens(compressed_opinion)
    print(f"[{reviewer.get_actor_name()}] done")

# Rounds 2 and 3
all_extractive_agents = extractive_reviewers + [extractive_author, extractive_meta]

extractive_discussion = Discussion(
    next_turn_manager=QueueTurnManager(actors=all_extractive_agents),
    users=all_extractive_agents,
    seed_opinions=extractive_opinions,
    seed_opinion_usernames=extractive_agent_names,
    history_context_len=6,
    conv_len=4,
)

extractive_discussion.begin(verbose=True)

for entry in extractive_discussion.get_logs():
    total_tokens += count_tokens(entry["text"])

extractive_meta_review = extractive_discussion.get_logs()[-1]["text"]
extractive_total_tokens = total_tokens

[StatisticalReviewer] done
[NLPReviewer] done
[SkepticalReviewer] done


  0%|          | 0/4 [00:00<?, ?it/s]

Comment by user StatisticalReviewer: "
Based on the abstract and the comments provided, the Transformer model indeed presents a compelling and innovative approach to sequence transduction tasks. Here are some additional points and considerations for a comprehensive review:  ### Model Architecture The Transformer architecture, as described, is remarkable for its reliance solely on attention mechanisms. This approach fundamentally shifts the paradigm away from sequential processing, which is a hallmark of traditional RNNs and CNNs. The use of self-attention and QKV mechanisms allows the model to efficiently compute pairwise interactions between different positions in the input sequence, thereby capturing dependencies without the need for explicit recurrence or convolution.  ### Performance Evaluation While the abstract highlights the superiority of Transformer-based models, a rigorous statistical evaluation is essential to substantiate these claims. Comparisons with state-of-the-art mode

In [19]:
extractive_bertscore = compute_bertscore(extractive_meta_review, baseline_meta)

divider = "=" * 60
print(divider)
print("EXTRACTIVE — final review")
print(divider)
print(extractive_meta_review)
print()
print(f"Tokens (baseline):    {baseline_total_tokens}")
print(f"Tokens (extractive):  {extractive_total_tokens}")
print(f"BERTScore F1:         {extractive_bertscore['f1']:.4f}")

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

EXTRACTIVE — final review
Thank you for your thoughtful feedback. I appreciate the constructive criticism and will address each point with care:

### Model Architecture

The Transformer model indeed relies solely on attention mechanisms, which fundamentally differ from traditional RNNs and CNNs. The use of self-attention and QKV mechanisms allows the model to capture dependencies between elements in a sequence without sequential processing. This shift from sequential to parallel computation significantly reduces computational complexity and memory requirements, making the model more scalable and efficient.

### Performance Evaluation

While the abstract highlights the superiority of Transformer-based models, a rigorous statistical evaluation is essential. Comparisons with state-of-the-art models on benchmark tasks such as language modeling, machine translation, and other sequence-to-sequence tasks would provide compelling evidence. Additionally, ablation studies could help identify the

Let's revisit the checklist, this time for the extractive summarization pipeline.

---

* [ ] **On track?**
  Does the response actually address the question?

* [ ] **Clear + readable?**
  Is it easy to follow, or a bit messy?

* [ ] **Some real thinking?**
  Does it go beyond obvious or generic points?

* [ ] **Balanced?**
  Does it consider more than one angle, or just present a single take?

* [ ] **Any pushback?**
  Does it question assumptions or just accept them?

* [ ] **Feels complete?**
  Do you feel like you got a useful answer, or something partial/shallow?

* [ ] **Feels realistic?**
  Does this read like an actual human-generated review? Could you discern between the two?

* [ ] **Overall feel**
  Does this feel genuinely helpful for your task?

* [ ] **Most importantly**
  Did you actually get any insights on the paper?

---

Three compression methods, three different philosophies. Caveman strips words within
sentences. LLMLingua strips tokens within sentences. Extractive drops entire sentences.

Look at the numbers. Which method saved the most tokens? Which drifted the least from the
baseline? And most importantly, look back at your checklists. Did any of the compressed
reviews actually lose something that mattered, or did they hold up?

There is no right answer here. The best compression method depends on the input, the task,
and how much quality you are willing to trade for efficiency.

In [20]:
print(f"Tokens (baseline):    {baseline_total_tokens}")
print(f"Tokens (caveman):     {caveman_total_tokens}")
print(f"Tokens (llmlingua):   {lingua_total_tokens}")
print(f"Tokens (extractive):  {extractive_total_tokens}")
print()
print(f"BERTScore F1 (caveman):    {bertscore_result['f1']:.4f}")
print(f"BERTScore F1 (llmlingua):  {lingua_bertscore['f1']:.4f}")
print(f"BERTScore F1 (extractive): {extractive_bertscore['f1']:.4f}")

Tokens (baseline):    8302
Tokens (caveman):     6741
Tokens (llmlingua):   5807
Tokens (extractive):  6896

BERTScore F1 (caveman):    0.8737
BERTScore F1 (llmlingua):  0.8481
BERTScore F1 (extractive): 0.8400


### Your turn

You now have three compression methods and a way to measure their impact. Here are some
things to try:

- **Compress what the agents say, not just what they receive.** So far we only compressed
  the outputs after the models generated them. But most of the tokens come from generation
  itself. Could you use prompting to affect how much the agents say in the first place?
- **Push LLMLingua harder.** We used `rate=0.5`. What happens at `rate=0.1`? At what point
  does the review stop making sense?
- **Feed more than the abstract.** We only used the abstract because it fits comfortably in
  context. What if you add the introduction? The methodology? At what point does compression
  become necessary rather than optional?
- **Combine methods.** What if you caveman-compress the input and LLMLingua-compress the
  discussion history? Or the other way around?
- **Use an LLM to summarize.** We used extractive summarization which needs no model. What
  if you used the LLM itself to summarize the discussion history before feeding it back?
  What are the tradeoffs?